# 🚀 LLM Gateway — Build With LiteLLM

In [2]:
!pip install litellm langchain-litellm langchain openai langchain-aws -q

In [3]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [4]:
import litellm
litellm.suppress_debug_info = True

## The Simplest LiteLLM Example — Unified API
The biggest pain point: **every provider has a** **different SDK.**

LiteLLM gives you **one function** — completion() — that works with all of them. Look at how clean this is:

In [13]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_BASE_URL"] = "https://bedrock-mantle.us-east-1.api.aws/v1"

os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["GROQ_API_KEY"] = userdata.get('Groq')
# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
Anthropic key loaded:  ❌
Groq key loaded:       ✅


In [22]:
from litellm import completion

# Same code, different providers — just change the `model` string!
# Call OpenAI through LiteLLM with explicit provider prefix

response_openai = completion(
    model="openai/openai.gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Write a one-sentence story about a unicorn."
        }
    ]
)

print("🔵 OpenAI:", response_openai.choices[0].message.content)

🔵 OpenAI: Under the silver glow of the moon, the gentle unicorn tiptoed across a meadow of whispering lavender, tucked a glowing star into its mane, and drifted into a soft, dreamy slumber, leaving a trail of twinkling sighs that lulled the night to sleep.


In [51]:
# Call qwen3.6 through Groq
response_groq = completion(
    model="groq/qwen/qwen3.6-27b",
    messages=[
        {"role": "system", "content": "Respond directly. Do not include any internal thought process or <think> tags. Provide ONLY the answer."},
        {"role": "user", "content": "Explain RAG in one sentence."}
    ]
)

# print("🟢 Groq (Qwen Direct):", response_groq.choices[0].message.content)

In [53]:
from litellm import completion

# Call GPT-OSS 20B through Groq
response_groq = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in one sentence."
        }
    ]
)

print("🟢 Groq:", response_groq.choices[0].message.content)

🟢 Groq: RAG (Retrieval‑Augmented Generation) is a hybrid language‑model approach that retrieves relevant external documents during inference and uses them as additional input to generate more informed, accurate responses.


In [55]:
from litellm import completion

prompt = "Explain AI in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "openai/openai.gpt-oss-120b"),
    ("🟢 Groq",       "groq/openai/gpt-oss-20b"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : Artificial intelligence is the branch of computer science that develops systems 
🟢 Groq         : Artificial Intelligence is a branch of computer science that enables machines to
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : ❌ APIConnectionError


# Automatic Fallbacks
**Real story:** OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [57]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "openai/openai.gpt-oss-120b",
        "groq/openai/gpt-oss-20b"
    ]
)

print("✅ App still got a response, even though the primary failed!")
print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

18:50:52 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.APIConnectionError: Missing Gemini API key. Set the GEMINI_API_KEY or GOOGLE_API_KEY environment variable.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/litellm/main.py", line 639, in acompletion
    response = await init_response
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/vertex_ai/gemini/vertex_and_google_ai_studio_gemini.py", line 2784, in async_completion
    auth_header, api_base = self._get_token_and_url(
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 702, in _get_token_and_url
    raise ValueError(
ValueError: Missing Gemini API key. Set the GEMINI_API_KEY or GOOGLE_API_KEY environment variable.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-package

Response: ## In a nutshell  

**An LLM Gateway is a dedicated middleware layer that sits between your applications (or users) and one or more Large Language Model (LLM) providers.**  
It exposes a stable, polic ...

Which model actually answered? openai.gpt-oss-120b


If **one-model** is rate-limited or down, LiteLLM transparently retries with Claude, then Groq. Your app **never sees the failure**.

This is the #1 reason teams adopt an LLM Gateway.

# Cost Tracking — Know Where Your Money Goes
LiteLLM **automatically calculates the cost** of every call using its built-in pricing database. No more surprise bills.

In [67]:
from litellm import completion

response = completion(
    model="openai/openai.gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Write a haiku about AI."}
    ]
)

print("Response:     ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)

Response:      Electric whisper,  
Learning the shape of language—  
Dreams of steel sunrise.

Input tokens:  74
Output tokens: 253


In [71]:
from litellm import completion, completion_cost

response = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Write a haiku about AI."}
    ]
)

# Groq GPT-OSS 20B pricing:
# Input: $0.075 / 1M tokens
# Output: $0.30 / 1M tokens
input_cost = response.usage.prompt_tokens * 0.075 / 1_000_000
output_cost = response.usage.completion_tokens * 0.30 / 1_000_000
cost = input_cost + output_cost

print("Response:     ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:      Metal heart whispers,  
Binary thoughts swirl in dark,  
Dawn of new bright mind.

Input tokens:  78
Output tokens: 310
Cost:         $0.00009885


# Smart Routing — The Right Model for the Right Job
**Why use one model for everything?**

      Coding tasks → Claude Sonnet
      Cheap summaries → GPT-4o-mini
      Super fast replies → Groq Llama
      Complex reasoning → Claude Opus
Use LiteLLM's **Router** to define routing rules:

In [74]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              # 👈 alias kept
        "litellm_params": {
            "model": "openai/openai.gpt-oss-120b",                # 👈 mapped to OpenAI instead
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "openai/openai.gpt-oss-20b",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (GPT):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  **AI is changing software – a quick summary**

1. **Automated code generation** – Large Language Models (LLMs) like GPT‑4 can produce functional code 

🧠 Smart/coding (GPT-4o):
 Here’s a concise, reusable function that reverses a string in Python.  
It works for ordinary ASCII as well as Unicode characters (including emojis, accented letters, etc.) because it operates on the string’s *code points* rather than raw bytes.

```python
def reverse_string(s: str) -> str:
    """



**💡 Key insight:** Your app calls "**fast-cheap" or "smart-coding"** — abstract names. The router decides which provider to actually use. Tomorrow, you can swap Groq for a cheaper provider with **zero code changes.**

# Load Balancing Across Multiple API Keys
Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [76]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "openai/openai.gpt-oss-120b",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt"}
    },

    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq/openai/gpt-oss-20b"}
    },
]
router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        groq/openai/gpt-oss-20b   407 ms   Hello! How can I assist you today?
#2        groq/openai/gpt-oss-20b   696 ms   Hello! What’s your next request?
#3        groq/openai/gpt-oss-20b   662 ms   Hello! Could you please provide me 
#4        openai-gpt              2708 ms   Hello! Could you please let me know
#5        openai-gpt              1701 ms   Hello! 👋 Could you let me know what
#6        groq/openai/gpt-oss-20b   778 ms   Hello! Could you please provide six


# 🎯 Strategy 1: least-busy
The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [77]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "openai/openai.gpt-oss-120b",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 OpenAI"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-20b",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   # 👈 the magic
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )

    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🔵 OpenAI
Request 2 → 🔵 OpenAI
Request 3 → 🔵 OpenAI
Request 4 → 🔵 OpenAI
Request 5 → 🔵 OpenAI
Request 6 → 🔵 OpenAI
Request 7 → 🔵 OpenAI
Request 8 → 🔵 OpenAI

🎯 Distribution:
   🔵 OpenAI: ████████ (8)


#**Integrating the Gateway with LangChain**
Here's where it really clicks for production GenAI apps:

**LangChain** for the orchestration (agents, chains, RAG) + **LiteLLM** as the unified LLM backend.

LangChain has a built-in **ChatLiteLLM** wrapper — drop it in like any other chat model.

In [78]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="openai/openai.gpt-oss-120b", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishGPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

- **Unified entry point** – a thin API layer that routes requests to one or multiple large language models (e.g., OpenAI, Anthropic, local models) so developers interact with a single, consistent interface.  
- **Cross‑model orchestration** – handles model selection, prompt routing, fallback, and aggregation of responses, enabling hybrid pipelines (e.g., choose the cheapest model for simple tasks, a more capable model for complex ones).  
- **Operational safeguards** – adds authentication, rate‑limiting, logging, content filtering, and cost‑monitoring, providing security, observability, and governance without modifying the downstream LLMs.


**🎯 The magic:** swap model="gpt-4o-mini" → "claude-3-5-sonnet-20241022" → "groq/llama-3.3-70b-versatile" and the entire chain now runs on a different provider. Zero other changes.

# **A Real Example — Multi-Provider LangChain Chain with Fallbacks**
Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.

In [79]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="openai/openai.gpt-oss-120b", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/openai/gpt-oss-20b", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

{"answer":"1. **Unified Access & Management** – A single gateway abstracts away the complexities of multiple LLM providers, offering a consistent API, centralized authentication, usage monitoring, and policy enforcement across models.\n2. **Optimized Cost & Performance** – The gateway can route requests to the most appropriate model based on latency, token pricing, or workload characteristics, enabling dynamic model selection, caching, and batching to reduce expenses and improve response times.\n3. **Enhanced Security & Compliance** – By acting as a controlled entry point, the gateway enforces data sanitization, encryption, audit logging, and compliance checks (e.g., GDPR, HIPAA), ensuring that sensitive information is handled safely before reaching any LLM."}


# A Mini End-to-End Demo — Smart Router for a Chatbot
Let's build a tiny **task-aware chatbot** that

      Decides what kind of question the user is asking (code, summary, general)

      Routes to the right model accordingly

      Falls back if the chosen model fails

      Logs cost and latency

In [100]:
import time
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="openai/qwen.qwen3-32b",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()


def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error

def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    ["openai/openai.gpt-oss-120b",                     "gpt-4o-mini",   "groq/openai/gpt-oss-20b"],
        "summary": ["gpt-4o-mini",                "groq/openai/gpt-oss-20b", "openai/qwen.qwen3-32b"],
        "general": ["groq/openai/gpt-oss-20b", "openai/qwen.qwen3-32b"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }
  # Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")


❓ Q: Write a Python function to compute Fibonacci numbers.
🏷️  Task:    code
🤖 Model:    openai.gpt-oss-120b
⏱️  Latency: 7.59s
💰 Cost:    $0.000000
💬 Answer:  Below is a compact, well‑documented Python implementation that lets you compute Fibonacci numbers in three common ways:

* **Iterative (O(n) time, O(1) space)** – the fastest and most memory‑efficient...
❓ Q: Summarize the importance of attention mechanism in 2 sentences.
   ⚠️  gpt-4o-mini failed (BadRequestError), trying next...
🏷️  Task:    summary
🤖 Model:    openai/gpt-oss-20b
⏱️  Latency: 0.52s
💰 Cost:    n/a
💬 Answer:  The attention mechanism allows models to dynamically weight and focus on the most relevant parts of input data, thereby improving interpretability and performance across diverse tasks such as language...
❓ Q: Tell me a fun fact about elephants.
🏷️  Task:    general
🤖 Model:    openai/gpt-oss-20b
⏱️  Latency: 0.4s
💰 Cost:    n/a
💬 Answer:  Did you know that elephants can “hear” vibrations in the ground from 